In [1]:
import os 
import cv2 
import glob 
import numpy as np 

import torch 
from pathlib import Path 
from torchvision.transforms.functional import to_tensor 
import onnxruntime as ort 


parent_dir = os.getcwd()
print(parent_dir)    

dataset_path = os.path.join(parent_dir, "samples/training_images")
image_paths = []
for file in os.listdir(dataset_path): 
    file_path = os.path.join(dataset_path, file)

    if not file.endswith("jpg") or file.endswith("png"):
        print(file_path)
        os.remove(file_path) 
        continue 

    image_paths.append(file_path)

/home/icsa_jim/Object_Tracking_ROI/EDGEAI_OBS_ROI/video_processing_system


In [2]:
import glob
from DepthAnythingV2.depth_anything_v2.dpt import DepthAnythingV2 
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # or 'vits', 'vitb', 'vitg'

xFormers not available
xFormers not available


In [3]:
depth_anything_path = parent_dir + '/DepthAnythingV2'
model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(f'{depth_anything_path}/checkpoints/depth_anything_v2_{encoder}.pth', map_location='cpu', weights_only=True))
model = model.to(DEVICE).eval()

In [4]:
from Depth_Anything_ONNX.depth_anything.util.transform import load_image
import time 

onnx_path = os.path.join(parent_dir, "Depth_Anything_ONNX/weights/depth_anything_vits14.onnx")
sess_options = ort.SessionOptions()
sess_options.enable_profiling = False
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] 

session = ort.InferenceSession(onnx_path, sess_options=sess_options, providers=providers)
image_file_path = os.getcwd() + '/DepthAnythingV2/assets/examples/demo01.jpg'
image = cv2.imread(str(image_file_path))
frame, (orig_h, orig_w) = load_image(image)


In [5]:
cv2.imshow('Depth Image', image)

In [6]:
# def benchmark_onnx(session, input_tensor_np, input_name, runs=50):
#     # Warm-up
#     for _ in range(5):
#         session.run(None, {input_name: input_tensor_np})

#     start = time.time()
#     for _ in range(runs):
#         session.run(None, {input_name: input_tensor_np})
#     end = time.time()

#     latency = (end - start) / runs * 1000
#     fps = 1000 / latency
#     return latency, fps

def benchmark_onnx(session, input_tensor_np, input_name,batch_size):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    with torch.no_grad():
        for i in range(0, len(input_tensor_np)):
            aspectRatio = orig_w / orig_h
            depth_height  = 518 #fixed 
            depth_width = round(depth_height * aspectRatio / 14) * 14
            depth_width = (depth_width // 14) * 14
            _ = session.run(None, {image:input_tensor_np[i]})
    end.record()
    torch.cuda.synchronize()
    latency = start.elapsed_time(end)
    throughput = len(input_tensor_np) / latency
    return throughput

In [7]:
# def benchmark_pytorch(model, input_tensor, org_w, orig_h, runs=50):
#     aspectRatio = org_w / orig_h
#     depth_height  = 518 #fixed 
#     depth_width = round(depth_height * aspectRatio / 14) * 14
#     depth_width = (depth_width // 14) * 14
#     with torch.no_grad():
#         for _ in range(5):
#             _ = model.infer_image(input_tensor, 518,precision=f"fp32", depthHeight=depth_height, depthWidth=depth_width)


#         start = time.time()
#         for _ in range(runs):
#             _ = model.infer_image(input_tensor, 518,precision=f"fp32", depthHeight=depth_height, depthWidth=depth_width)

#         end = time.time()

#     latency = (end - start) / runs * 1000
#     fps = 1000 / latency
#     return latency, fps

def benchmark_pytorch(model, inputs, batch_size,orig_w, orig_h):
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    with torch.no_grad():
        for i in range(0, len(inputs)):
            aspectRatio = orig_w / orig_h
            depth_height  = 518 #fixed 
            depth_width = round(depth_height * aspectRatio / 14) * 14
            depth_width = (depth_width // 14) * 14
            _ = model.infer_image(inputs[i], 518,precision=f"fp32", depthHeight=depth_height, depthWidth=depth_width)
    end.record()
    torch.cuda.synchronize()
    latency = start.elapsed_time(end)
    throughput = len(inputs) * batch_size / latency
    return throughput


In [8]:
input_name = session.get_inputs()[0].name

In [9]:
image_dir = parent_dir + "/DepthAnythingV2/assets/examples/"
lat_pt,lat_onnx, fps_pt, fps_onnx = 0.0,0.0,0.0,0.0 
images_raw = [] 
frames = [] 
for image_path in os.listdir(image_dir): 
    image_path = os.path.join(image_dir, image_path)
    if image_path.endswith(".png"):
        pass 
    image = cv2.imread(str(image_file_path))
    images_raw.append(image)
    frames.append(image)
    frame, (orig_h, orig_w) = load_image(image)


In [10]:
print(frames[0].astype(np.float32))


[[[ 93.  76.  79.]
  [106.  88.  89.]
  [ 96.  75.  73.]
  ...
  [247. 214. 175.]
  [247. 214. 175.]
  [247. 214. 175.]]

 [[ 92.  75.  78.]
  [108.  90.  91.]
  [ 99.  78.  76.]
  ...
  [247. 214. 175.]
  [247. 214. 175.]
  [247. 214. 175.]]

 [[ 86.  69.  72.]
  [106.  88.  89.]
  [ 99.  78.  76.]
  ...
  [247. 214. 175.]
  [247. 214. 175.]
  [247. 214. 175.]]

 ...

 [[ 62.  43.  35.]
  [ 64.  45.  37.]
  [ 64.  45.  37.]
  ...
  [189. 144. 110.]
  [189. 144. 111.]
  [190. 145. 111.]]

 [[ 60.  41.  33.]
  [ 61.  42.  34.]
  [ 61.  42.  34.]
  ...
  [189. 142. 110.]
  [190. 143. 112.]
  [191. 144. 112.]]

 [[ 61.  42.  34.]
  [ 62.  43.  35.]
  [ 61.  42.  34.]
  ...
  [189. 142. 111.]
  [190. 143. 112.]
  [191. 144. 113.]]]


In [11]:
batch_size = 16 

throughput_pt = benchmark_pytorch(model, images_raw,batch_size, orig_w, orig_h)
throughput_onnx = benchmark_onnx(session, frames, input_name,batch_size)
# temp_lat_pt, temp_fps_pt = benchmark_pytorch(model, image, orig_w, orig_h)
# temp_lat_onnx,temp_fps_onnx = benchmark_onnx(session, frame, input_name)

# lat_pt += temp_lat_pt
# fps_pt += temp_fps_pt
# lat_onnx += temp_lat_onnx
# fps_onnx += temp_fps_onnx

# lat_pt /= len(image_paths)
# lat_onnx /= len(image_paths)
# fps_pt /= len(image_paths)
# fps_onnx /= len(image_paths)

TypeError: unhashable type: 'numpy.ndarray'

In [ ]:
# === PRINT RESULTS ===
print(f"\n🧠 PyTorch Model: Latency = {lat_pt:.2f} ms | FPS = {fps_pt:.2f}")
print(f"🚀 ONNX Model:    Latency = {lat_onnx:.2f} ms | FPS = {fps_onnx:.2f}")

In [ ]:
def measure_gpu_throughput(model, inputs, batch_size):
    inputs = inputs.to('cuda')
    model = model.to('cuda')
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    with torch.no_grad():
        for i in range(0, inputs.size(0), batch_size):
            output = model(inputs[i:i + batch_size])
    end.record()
    torch.cuda.synchronize()
    latency = start.elapsed_time(end)
    throughput = inputs.size(0) * batch_size / latency
    return throughput

In [ ]:
# from torch.ao.quantization import quantize_dynamic
# # This is Post Training Dynamic Quantization 
# quantized = quantize_dynamic(model, {torch.nn.Linear, torch.nn.Conv2d}, dtype=torch.qint8)
# quantized.eval()
# model.eval()

In [ ]:
# def get_model_size(model_path):
#     size_mb = os.path.getsize(model_path) / (1024 * 1024)
#     return round(size_mb, 2)
# num_params = len(torch.nn.utils.parameters_to_vector(model.parameters()))
# print(f"Number of parameters for normal model: {num_params}")
# qnum_params = len(torch.nn.utils.parameters_to_vector(quantized.parameters()))
# print(f"Number of parameters for quantized model: {qnum_params}")
# percent = (num_params - qnum_params) / num_params * 100
# print(f"Percent reduction in parameters: {percent:.2f}%")


In [ ]:
image_file_path = os.getcwd() + '/samples/training_images/vid_4_600.jpg'
input_tensor = load_sample_image(image_file_path,size=(518,518),device=DEVICE)


In [ ]:
lat1, fps1 = measure_inference_speed(model, input_tensor)
print("\n⚙️ Inference Performance:")
print(f"Model 1 Latency: {lat1} ms, FPS: {fps1}")
q_input = input_tensor.to("cpu")
quantized = quantized.to("cpu")
lat2, fps2 = measure_inference_speed(quantized, input_tensor)

print(f"Model 2 Latency: {lat2} ms, FPS: {fps2}")


In [ ]:

mse = evaluate_output_accuracy(model,quantized , input_tensor)
print(f"\n🎯 Output MSE (as proxy for accuracy difference): {mse:.6f}")

In [ ]:
video_path = "samples/sample_video.mp4" 
filename = "sample_video.mp4"
raw_video = cv2.VideoCapture(video_path)
frame_width, frame_height = int(raw_video.get(cv2.CAP_PROP_FRAME_WIDTH)), int(raw_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_rate = int(raw_video.get(cv2.CAP_PROP_FPS))


In [ ]:
import matplotlib

margin_width = 50
cmap = matplotlib.colormaps.get_cmap('Spectral_r')

In [ ]:

pred_only = True 
if pred_only:
    output_width = frame_width 
else: 
    output_width = frame_width * 2 + margin_width

In [ ]:
output_path = os.path.join(parent_dir, "samples/sample_video_depth.mp4")
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), frame_rate, (output_width, frame_height))


In [ ]:
from enum import Enum, auto

class StrEnum(str, Enum):
    pass

class InferenceDevice(StrEnum):
    cpu = auto()
    cuda = auto()
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

if DEVICE == "cuda": 
    device = InferenceDevice.cuda
else: 
    device = InferenceDevice.cpu 

In [ ]:

grayscale = False 
while raw_video.isOpened(): 
    ret, image = raw_video.read()
    if not ret:
        break

    frame, (orig_h, orig_w) = load_image(image)
    depth = session.run(None, {"image": frame})[0]
    depth = cv2.resize(depth[0, 0], (orig_w, orig_h))

    depth = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
    depth = depth.astype(np.uint8)
    depth_color = cv2.applyColorMap(depth, cv2.COLORMAP_INFERNO)
    margin_width = 50
    caption_height = 60
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    font_thickness = 2
    split_region = np.ones((orig_h, margin_width, 3), dtype=np.uint8) * 255
    combined_results = cv2.hconcat([image, split_region, depth_color])

    caption_space = (
        np.ones((caption_height, combined_results.shape[1], 3), dtype=np.uint8)
        * 255
    )
    captions = ["Raw image", "Depth Anything"]
    segment_width = orig_w + margin_width
    for i, caption in enumerate(captions):
        # Calculate text size
        text_size = cv2.getTextSize(caption, font, font_scale, font_thickness)[0]

        # Calculate x-coordinate to center the text
        text_x = int((segment_width * i) + (orig_w - text_size[0]) / 2)

        # Add text caption
        cv2.putText(
            caption_space,
            caption,
            (text_x, 40),
            font,
            font_scale,
            (0, 0, 0),
            font_thickness,
        )

    final_result = cv2.vconcat([caption_space, combined_results])

    cv2.imshow("depth", final_result)
    cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
grayscale = False
while raw_video.isOpened():
    ret, raw_frame = raw_video.read() 
    if not ret:
        break 
    
    
    depth = model.infer_image(raw_frame, 518)
    depth = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
    depth = depth.astype(np.uint8)
    if grayscale:
        depth = np.repeat(depth[..., np.newaxis], 3, axis=-1)
    else:
        depth = (cmap(depth)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
    
    if pred_only:
        out.write(depth)
    else:
        split_region = np.ones((frame_height, margin_width, 3), dtype=np.uint8) * 255
        combined_frame = cv2.hconcat([raw_frame, split_region, depth])
        
        out.write(combined_frame)

raw_video.release()
out.release()

In [ ]:
parent_dir